# Bina OCR - Batch PDF/Image OCR on Kaggle

Uses [Reza2kn/Bina-0.1](https://huggingface.co/Reza2kn/Bina-0.1) (Persian OCR vision-language model).

**How to use:**
1. Upload your PDF or image folder as a Kaggle Dataset
2. Add the dataset to this notebook (Add data → search your dataset)
3. Set `INPUT_PATH` below to the dataset path (e.g. `/kaggle/input/my-pdf-dataset`)
4. Run all cells

In [ ]:
# ============================================================
# CONFIG - Edit these before running
# ============================================================

# Path to your PDF file or image folder (Kaggle dataset path)
INPUT_PATH = "/kaggle/input/your-dataset-name/your_file.pdf"  # <-- CHANGE THIS

# Set to "pdf" or "images"
INPUT_TYPE = "pdf"

# Output settings
OUTPUT_DIR = "/kaggle/working"
OUTPUT_FILE = "book_transcript.md"
TIMING_LOG = "page_timings.csv"

# Model settings
MODEL_ID = "Reza2kn/Bina-0.1"
PROMPT_TEXT = "Transcribe all text in this image exactly as it appears."
PDF_DPI = 200          # Lower = faster (150-300)
MAX_NEW_TOKENS = 512   # Max tokens generated per page
PAGE_LIMIT = None      # Set to int to limit pages, None = all

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

## 1. Install Dependencies

In [ ]:
!pip install --force-reinstall git+https://github.com/huggingface/transformers.git pymupdf accelerate safetensors 2>&1 | grep -E "^(Successfully|ERROR|Collecting qwen)"
import transformers
print(f"transformers version: {transformers.__version__}")

# Verify qwen3_5 is recognized
from transformers import AutoConfig
cfg = AutoConfig.from_pretrained("Reza2kn/Bina-0.1")
print(f"Architecture: {cfg.model_type} - OK!")

**IMPORTANT: After running the install cell above, go to `Runtime → Restart session` before running any other cells!** The old transformers module is cached in memory and won't be replaced until you restart.

## 2. Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    device = "cuda"
else:
    print("No GPU found - will use CPU (very slow)")
    device = "cpu"

dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Using device: {device}, dtype: {dtype}")

## 3. Load Model

In [ ]:
import time
from transformers import AutoModelForMultimodalLM, AutoProcessor

print(f"Loading processor for {MODEL_ID} ...")
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(f"Processor loaded in {time.time() - t0:.1f}s")

print(f"Loading model weights (dtype={dtype}) ...")
t0 = time.time()
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map="auto" if device == "cuda" else None,
)
if device == "cpu":
    model = model.to(device)
model.eval()
print(f"Model loaded in {time.time() - t0:.1f}s on {model.device}")

## 4. Collect Pages

In [ ]:
import tempfile
from pathlib import Path
import fitz  # PyMuPDF

input_path = Path(INPUT_PATH)
pdf_tmp_dir = None

if INPUT_TYPE == "pdf":
    assert input_path.is_file(), f"PDF not found: {input_path}"
    pdf_tmp_dir = Path(tempfile.mkdtemp(prefix="ocr_pdf_"))
    print(f"Rendering PDF at {PDF_DPI} DPI...")
    doc = fitz.open(input_path)
    print(f"PDF has {len(doc)} pages")
    pages = []
    for i, page in enumerate(doc):
        pix = page.get_pixmap(dpi=PDF_DPI)
        out = pdf_tmp_dir / f"page_{i + 1:04d}.png"
        pix.save(out)
        pages.append(out)
    doc.close()
else:
    assert input_path.is_dir(), f"Folder not found: {input_path}"
    pages = sorted(p for p in input_path.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
    assert pages, f"No images found in {input_path}"

if PAGE_LIMIT:
    pages = pages[:PAGE_LIMIT]

print(f"Will process {len(pages)} pages")

## 5. Run OCR

In [ ]:
from tqdm import tqdm
from PIL import Image

output_path = Path(OUTPUT_DIR) / OUTPUT_FILE
timing_path = Path(OUTPUT_DIR) / TIMING_LOG

timings = []
total_start = time.time()

with open(output_path, "w", encoding="utf-8") as out_f:
    for i, page_path in enumerate(tqdm(pages, desc="OCR", unit="page"), start=1):
        page_start = time.time()

        try:
            image = Image.open(page_path).convert("RGB")
            messages = [{
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": PROMPT_TEXT},
                ],
            }]
            inputs = processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
                tokenize=True,
                return_dict=True,
                return_tensors="pt",
            ).to(model.device)

            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)

            generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
            text = processor.decode(generated_ids, skip_special_tokens=True).strip()
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            text = "[ERROR: out of memory, page skipped]"
        except Exception as e:
            text = f"[ERROR: {e}]"

        elapsed = time.time() - page_start
        timings.append((page_path.name, round(elapsed, 2)))

        out_f.write(f"## Page {i}: {page_path.name}\n\n{text}\n\n")
        out_f.flush()

        if i % 10 == 0 and device == "cuda":
            torch.cuda.empty_cache()

total_elapsed = time.time() - total_start

## 6. Summary & Cleanup

In [ ]:
import csv

# Save timing log
with open(timing_path, "w", newline="", encoding="utf-8") as csv_f:
    writer = csv.writer(csv_f)
    writer.writerow(["page_file", "seconds"])
    writer.writerows(timings)

# Cleanup temp PDF renders
if pdf_tmp_dir:
    for f in pdf_tmp_dir.iterdir():
        f.unlink()
    pdf_tmp_dir.rmdir()

# Print summary
avg_time = sum(t for _, t in timings) / len(timings)
print("\n--- Summary ---")
print(f"Pages processed: {len(timings)}")
print(f"Total time: {total_elapsed:.2f}s")
print(f"Average time/page: {avg_time:.2f}s")
print(f"Transcript: {output_path}")
print(f"Timing log: {timing_path}")
print(f"\nDownload from: {OUTPUT_DIR} (right-click → Download)" )

## 7. Preview Output

In [ ]:
# Show first 2000 chars of transcript
if output_path.exists():
    content = output_path.read_text(encoding="utf-8")
    print(content[:2000])
    if len(content) > 2000:
        print(f"\n... ({len(content)} total chars)")
else:
    print("No output yet. Run the OCR cell first.")